# Classification Modeling

This notebook extends the return-prediction analysis from regression to
three-class directional classification.

For each selected asset-horizon pair, future returns are classified as
**DOWN**, **STATIONARY**, or **UP** using an asset-specific threshold
estimated exclusively from the training period.

The classification models are developed using the March 16–18 training
period and evaluated for model selection on March 19. March 20 remains
strictly held out for final evaluation.

In [1]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report
)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

## 1. Load Processed Data

The classification experiment reuses the processed 500 ms datasets created during
feature engineering. Each asset-day was constructed independently before export,
so rolling features and future targets do not cross overnight boundaries.

In [3]:
from pathlib import Path

PROCESSED_DIR = Path("../data/processed")

assets = ["TSLA", "AMZN", "BRK.B"]

train_dates = [
    "2026-03-16",
    "2026-03-17",
    "2026-03-18",
]

val_dates = [
    "2026-03-19",
]

test_dates = [
    "2026-03-20",
]


def load_processed_day(asset, date):
    safe_asset = asset.replace(".", "_")

    path = (
        PROCESSED_DIR
        / f"{safe_asset}_{date}.parquet"
    )

    return pd.read_parquet(path)


def combine_processed_dates(asset, dates):
    return pd.concat(
        [
            load_processed_day(asset, date)
            for date in dates
        ],
        axis=0
    )


splits = {}

for asset in assets:
    splits[asset] = {
        "train": combine_processed_dates(asset, train_dates),
        "val": combine_processed_dates(asset, val_dates),
        "test": combine_processed_dates(asset, test_dates),
    }

    print(
        asset,
        "Train:", splits[asset]["train"].shape,
        "Validation:", splits[asset]["val"].shape,
        "Test:", splits[asset]["test"].shape,
    )

TSLA Train: (140400, 75) Validation: (46800, 75) Test: (46800, 75)
AMZN Train: (140400, 75) Validation: (46800, 75) Test: (46800, 75)
BRK.B Train: (140400, 75) Validation: (46800, 75) Test: (46800, 75)


## 2. Classification Target Construction

Classification is performed for one representative horizon per asset:

- TSLA: 500 ms
- AMZN: 500 ms
- BRK.B: 5 s

For each asset, the threshold `tau` is defined as the 10th percentile of the
absolute nonzero future returns in the training period.

The thresholds are estimated using March 16–18 only and are then frozen for
validation and test evaluation.

In [4]:
classification_targets = {
    "TSLA": "target_500ms",
    "AMZN": "target_500ms",
    "BRK.B": "target_5s",
}

label_names = {
    -1: "DOWN",
     0: "STATIONARY",
     1: "UP",
}


def make_classification_target(returns, threshold):
    labels = pd.Series(
        np.nan,
        index=returns.index,
        dtype=float
    )

    valid = returns.notna()

    labels.loc[
        valid & (returns < -threshold)
    ] = -1

    labels.loc[
        valid
        & (returns >= -threshold)
        & (returns <= threshold)
    ] = 0

    labels.loc[
        valid & (returns > threshold)
    ] = 1

    return labels

In [5]:
classification_thresholds = {}

for asset, target in classification_targets.items():

    y_train = splits[asset]["train"][target].dropna()

    abs_nonzero = y_train[y_train != 0].abs()

    classification_thresholds[asset] = (
        abs_nonzero.quantile(0.10)
    )


print("Frozen classification thresholds:")

for asset, threshold in classification_thresholds.items():
    print(
        asset,
        f"{threshold:.10e}"
    )

Frozen classification thresholds:
TSLA 1.2559579505e-05
AMZN 2.3400297185e-05
BRK.B 1.0220925301e-05


In [6]:
for asset, target in classification_targets.items():

    threshold = classification_thresholds[asset]

    for split_name in ["train", "val", "test"]:

        df = splits[asset][split_name].copy()

        df["classification_target"] = (
            make_classification_target(
                df[target],
                threshold
            )
        )

        splits[asset][split_name] = df

In [7]:
distribution_rows = []

for asset in assets:
    for split_name in ["train", "val", "test"]:

        y = (
            splits[asset][split_name]
            ["classification_target"]
            .dropna()
        )

        percentages = (
            y.value_counts(normalize=True)
            * 100
        )

        distribution_rows.append({
            "asset": asset,
            "split": split_name,
            "DOWN_pct": percentages.get(-1, 0),
            "STATIONARY_pct": percentages.get(0, 0),
            "UP_pct": percentages.get(1, 0),
        })


classification_distribution = pd.DataFrame(
    distribution_rows
)

display(classification_distribution)

,asset,split,DOWN_pct,STATIONARY_pct,UP_pct
0,TSLA,train,34.140331,31.136705,34.722964
1,TSLA,val,42.703904,15.763157,41.532939
2,TSLA,test,42.300049,16.429838,41.270113
3,AMZN,train,27.868117,44.562206,27.569677
4,AMZN,val,34.823394,30.970747,34.205859
5,AMZN,test,34.237911,32.171628,33.590461
6,BRK.B,train,27.863504,47.643371,24.493125
7,BRK.B,val,34.357769,34.695448,30.946784
8,BRK.B,test,30.378286,41.938448,27.683266


## 3. Majority-Class Baseline

Before fitting predictive models, we establish a simple classification
baseline.

For each asset, the baseline always predicts the most frequent class in
the training set. The predicted class is determined exclusively from the
training period and then applied unchanged to the validation set.

Because the class distributions are not perfectly balanced, both ordinary
accuracy and balanced accuracy are reported. Macro F1 is also included to
measure performance across all three classes rather than emphasizing the
most frequent class.

In [29]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score
)

baseline_results = []

for asset in assets:

    train_df = splits[asset]["train"]
    val_df = splits[asset]["val"]

    # Use exactly the same eligible rows as the learned classifiers:
    # target must be defined and all model features must be available.
    train_mask = (
        train_df[feature_cols].notna().all(axis=1)
        & train_df["classification_target"].notna()
    )

    val_mask = (
        val_df[feature_cols].notna().all(axis=1)
        & val_df["classification_target"].notna()
    )

    y_train = train_df.loc[
        train_mask,
        "classification_target"
    ].astype(int)

    y_val = val_df.loc[
        val_mask,
        "classification_target"
    ].astype(int)

    # Majority class determined from training data only
    majority_class = y_train.value_counts().idxmax()

    # ZeroR: predict the training majority class for every validation row
    y_pred = np.full(
        len(y_val),
        majority_class
    )

    baseline_results.append({
        "asset": asset,
        "model": "ZeroR / Majority Class",
        "predicted_class": label_names[majority_class],
        "accuracy": accuracy_score(y_val, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_val, y_pred),
        "macro_f1": f1_score(
            y_val,
            y_pred,
            average="macro",
            zero_division=0
        )
    })


baseline_results_df = pd.DataFrame(baseline_results)

display(baseline_results_df)

,asset,model,predicted_class,accuracy,balanced_accuracy,macro_f1
0,TSLA,ZeroR / Majority Class,UP,0.415435,0.333333,0.195669
1,AMZN,ZeroR / Majority Class,STATIONARY,0.309998,0.333333,0.157760
2,BRK.B,ZeroR / Majority Class,STATIONARY,0.346672,0.333333,0.171619


## 4. Multinomial Logistic Regression

We next fit a multinomial logistic regression classifier using the same
microstructure features considered in the regression experiments.

Features are standardized using parameters estimated from the training data
only. The model is then evaluated on the validation day.

Performance is compared against the majority-class baseline using accuracy,
balanced accuracy, and macro F1.

In [11]:
feature_cols = [
    # LOB state
    "spread",
    "imbalance_l1",
    "imbalance_l5",
    "bid_depth_5",
    "ask_depth_5",
    "microprice_deviation",

    # Order flow / activity
    "event_count",
    "execution_count",
    "normalized_book_flow",
    "execution_imbalance",

    # Past returns
    "return_500ms",
    "return_1s",
    "return_2s",
    "return_5s",
    "return_10s",

    # Volatility / recent activity
    "volatility_5s",
    "volatility_30s",
    "avg_events_5s",
    "avg_executions_5s",
]

print("Number of features:", len(feature_cols))

Number of features: 19


In [12]:
for asset in assets:
    missing_cols = [
        col for col in feature_cols
        if col not in splits[asset]["train"].columns
    ]

    print(asset, "missing columns:", missing_cols)

TSLA missing columns: []
AMZN missing columns: []
BRK.B missing columns: []


In [13]:
logistic_results = []
logistic_models = {}

for asset in assets:

    train_df = splits[asset]["train"]
    val_df = splits[asset]["val"]

    # Keep only rows with complete features and a defined target
    train_mask = (
        train_df[feature_cols].notna().all(axis=1)
        & train_df["classification_target"].notna()
    )

    val_mask = (
        val_df[feature_cols].notna().all(axis=1)
        & val_df["classification_target"].notna()
    )

    X_train = train_df.loc[train_mask, feature_cols]
    y_train = train_df.loc[
        train_mask, "classification_target"
    ].astype(int)

    X_val = val_df.loc[val_mask, feature_cols]
    y_val = val_df.loc[
        val_mask, "classification_target"
    ].astype(int)

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(
            max_iter=1000,
            solver="lbfgs"
        ))
    ])

    model.fit(X_train, y_train)

    y_pred = model.predict(X_val)

    logistic_models[asset] = model

    logistic_results.append({
        "asset": asset,
        "model": "Logistic Regression",
        "accuracy": accuracy_score(y_val, y_pred),
        "balanced_accuracy": balanced_accuracy_score(
            y_val, y_pred
        ),
        "macro_f1": f1_score(
            y_val,
            y_pred,
            average="macro",
            zero_division=0
        )
    })


logistic_results_df = pd.DataFrame(logistic_results)

display(logistic_results_df)

,asset,model,accuracy,balanced_accuracy,macro_f1
0,TSLA,Logistic Regression,0.442842,0.398590,0.400777
1,AMZN,Logistic Regression,0.375447,0.391712,0.318907
2,BRK.B,Logistic Regression,0.398930,0.386429,0.312040


In [14]:
classification_comparison = pd.concat(
    [
        baseline_results_df[
            [
                "asset",
                "model",
                "accuracy",
                "balanced_accuracy",
                "macro_f1"
            ]
        ],
        logistic_results_df
    ],
    ignore_index=True
)

display(
    classification_comparison.sort_values(
        ["asset", "model"]
    )
)

,asset,model,accuracy,balanced_accuracy,macro_f1
4,AMZN,Logistic Regression,0.375447,0.391712,0.318907
1,AMZN,Majority Class,0.309707,0.333333,0.157647
5,BRK.B,Logistic Regression,0.398930,0.386429,0.312040
2,BRK.B,Majority Class,0.346954,0.333333,0.171723
3,TSLA,Logistic Regression,0.442842,0.398590,0.400777
0,TSLA,Majority Class,0.415329,0.333333,0.195634


## 5. Per-Class Performance

Although Logistic Regression outperforms the majority-class baseline for all
three assets, aggregate metrics do not reveal how performance is distributed
across DOWN, STATIONARY, and UP observations.

We therefore examine confusion matrices and class-specific precision, recall,
and F1 scores on the validation set.

In [15]:
for asset in assets:

    val_df = splits[asset]["val"]

    val_mask = (
        val_df[feature_cols].notna().all(axis=1)
        & val_df["classification_target"].notna()
    )

    X_val = val_df.loc[val_mask, feature_cols]
    y_val = val_df.loc[
        val_mask, "classification_target"
    ].astype(int)

    y_pred = logistic_models[asset].predict(X_val)

    print("=" * 70)
    print(asset)
    print("=" * 70)

    cm = confusion_matrix(
        y_val,
        y_pred,
        labels=[-1, 0, 1]
    )

    cm_df = pd.DataFrame(
        cm,
        index=["Actual DOWN", "Actual STATIONARY", "Actual UP"],
        columns=["Pred DOWN", "Pred STATIONARY", "Pred UP"]
    )

    display(cm_df)

    print(
        classification_report(
            y_val,
            y_pred,
            labels=[-1, 0, 1],
            target_names=["DOWN", "STATIONARY", "UP"],
            digits=4,
            zero_division=0
        )
    )

TSLA


,Pred DOWN,Pred STATIONARY,Pred UP
Actual DOWN,10349,2127,7470
Actual STATIONARY,3185,1712,2479
Actual UP,8640,2140,8637


              precision    recall  f1-score   support

        DOWN     0.4667    0.5189    0.4914     19946
  STATIONARY     0.2863    0.2321    0.2564      7376
          UP     0.4647    0.4448    0.4545     19417

    accuracy                         0.4428     46739
   macro avg     0.4059    0.3986    0.4008     46739
weighted avg     0.4374    0.4428    0.4390     46739

AMZN


,Pred DOWN,Pred STATIONARY,Pred UP
Actual DOWN,3205,11675,1388
Actual STATIONARY,1243,12513,733
Actual UP,2627,11525,1830


              precision    recall  f1-score   support

        DOWN     0.4530    0.1970    0.2746     16268
  STATIONARY     0.3504    0.8636    0.4985     14489
          UP     0.4632    0.1145    0.1836     15982

    accuracy                         0.3754     46739
   macro avg     0.4222    0.3917    0.3189     46739
weighted avg     0.4247    0.3754    0.3129     46739

BRK.B


,Pred DOWN,Pred STATIONARY,Pred UP
Actual DOWN,3391,11899,776
Actual STATIONARY,1479,14333,388
Actual UP,2617,10929,918


              precision    recall  f1-score   support

        DOWN     0.4529    0.2111    0.2879     16066
  STATIONARY     0.3857    0.8848    0.5372     16200
          UP     0.4409    0.0635    0.1110     14464

    accuracy                         0.3989     46730
   macro avg     0.4265    0.3864    0.3120     46730
weighted avg     0.4259    0.3989    0.3196     46730



## 6. Class-Balanced Logistic Regression

The initial Logistic Regression model performs better than the majority-class
baseline in balanced accuracy for all three assets. However, the confusion
matrices reveal substantial differences in class-specific performance.

For AMZN and BRK.B, the model strongly favors the STATIONARY class and has
relatively low recall for directional movements, particularly UP observations.
TSLA exhibits a more balanced prediction pattern, although STATIONARY remains
the most difficult class.

To investigate whether unequal class representation contributes to this
behavior, we next evaluate Logistic Regression with class-balanced loss
weights. Model comparison is performed exclusively on the validation set.

In [16]:
balanced_logistic_models = {}
balanced_logistic_results = []

for asset in assets:

    train_df = splits[asset]["train"]
    val_df = splits[asset]["val"]

    train_mask = (
        train_df[feature_cols].notna().all(axis=1)
        & train_df["classification_target"].notna()
    )

    val_mask = (
        val_df[feature_cols].notna().all(axis=1)
        & val_df["classification_target"].notna()
    )

    X_train = train_df.loc[train_mask, feature_cols]
    y_train = train_df.loc[
        train_mask, "classification_target"
    ].astype(int)

    X_val = val_df.loc[val_mask, feature_cols]
    y_val = val_df.loc[
        val_mask, "classification_target"
    ].astype(int)

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(
            max_iter=1000,
            class_weight="balanced"
        ))
    ])

    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)

    balanced_logistic_models[asset] = model

    balanced_logistic_results.append({
        "asset": asset,
        "model": "Balanced Logistic Regression",
        "accuracy": accuracy_score(y_val, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_val, y_pred),
        "macro_f1": f1_score(y_val, y_pred, average="macro")
    })

balanced_logistic_results_df = pd.DataFrame(
    balanced_logistic_results
)

display(balanced_logistic_results_df)

,asset,model,accuracy,balanced_accuracy,macro_f1
0,TSLA,Balanced Logistic Regression,0.424421,0.409530,0.400614
1,AMZN,Balanced Logistic Regression,0.424913,0.427740,0.419580
2,BRK.B,Balanced Logistic Regression,0.412883,0.410076,0.407589


In [18]:
classification_comparison = pd.concat(
    [
        logistic_results_df,
        balanced_logistic_results_df
    ],
    ignore_index=True
)

display(
    classification_comparison.sort_values(
        ["asset", "balanced_accuracy"],
        ascending=[True, False]
    )
)

,asset,model,accuracy,balanced_accuracy,macro_f1
4,AMZN,Balanced Logistic Regression,0.424913,0.427740,0.419580
1,AMZN,Logistic Regression,0.375447,0.391712,0.318907
5,BRK.B,Balanced Logistic Regression,0.412883,0.410076,0.407589
2,BRK.B,Logistic Regression,0.398930,0.386429,0.312040
3,TSLA,Balanced Logistic Regression,0.424421,0.409530,0.400614
0,TSLA,Logistic Regression,0.442842,0.398590,0.400777


In [19]:
for asset in assets:

    val_df = splits[asset]["val"]

    val_mask = (
        val_df[feature_cols].notna().all(axis=1)
        & val_df["classification_target"].notna()
    )

    X_val = val_df.loc[val_mask, feature_cols]
    y_val = val_df.loc[
        val_mask, "classification_target"
    ].astype(int)

    model = balanced_logistic_models[asset]
    y_pred = model.predict(X_val)

    cm = confusion_matrix(
        y_val,
        y_pred,
        labels=[-1, 0, 1]
    )

    cm_df = pd.DataFrame(
        cm,
        index=["Actual DOWN", "Actual STATIONARY", "Actual UP"],
        columns=["Pred DOWN", "Pred STATIONARY", "Pred UP"]
    )

    print("=" * 70)
    print(asset)
    print("=" * 70)

    display(cm_df)

    print(
        classification_report(
            y_val,
            y_pred,
            labels=[-1, 0, 1],
            target_names=["DOWN", "STATIONARY", "UP"],
            digits=4
        )
    )

TSLA


,Pred DOWN,Pred STATIONARY,Pred UP
Actual DOWN,9808,3753,6385
Actual STATIONARY,2788,2621,1967
Actual UP,8322,3687,7408


              precision    recall  f1-score   support

        DOWN     0.4689    0.4917    0.4800     19946
  STATIONARY     0.2605    0.3553    0.3006      7376
          UP     0.4701    0.3815    0.4212     19417

    accuracy                         0.4244     46739
   macro avg     0.3998    0.4095    0.4006     46739
weighted avg     0.4365    0.4244    0.4273     46739

AMZN


,Pred DOWN,Pred STATIONARY,Pred UP
Actual DOWN,7687,5175,3406
Actual STATIONARY,4362,7604,2523
Actual UP,6222,5191,4569


              precision    recall  f1-score   support

        DOWN     0.4207    0.4725    0.4451     16268
  STATIONARY     0.4231    0.5248    0.4685     14489
          UP     0.4352    0.2859    0.3451     15982

    accuracy                         0.4249     46739
   macro avg     0.4264    0.4277    0.4196     46739
weighted avg     0.4264    0.4249    0.4182     46739

BRK.B


,Pred DOWN,Pred STATIONARY,Pred UP
Actual DOWN,5966,5589,4511
Actual STATIONARY,3744,8446,4010
Actual UP,4471,5111,4882


              precision    recall  f1-score   support

        DOWN     0.4207    0.3713    0.3945     16066
  STATIONARY     0.4411    0.5214    0.4779     16200
          UP     0.3642    0.3375    0.3504     14464

    accuracy                         0.4129     46730
   macro avg     0.4087    0.4101    0.4076     46730
weighted avg     0.4103    0.4129    0.4098     46730



### Validation observations

Class-balanced Logistic Regression improves balanced accuracy for all three
assets and produces substantially more even class-specific recall than the
unweighted model.

The improvement is particularly clear for AMZN and BRK.B, where the original
classifier strongly favored the STATIONARY class. With class-balanced loss
weights, directional DOWN and UP observations receive substantially higher
recall while STATIONARY performance remains useful.

TSLA shows a smaller improvement in balanced accuracy, but its predictions
remain comparatively balanced across the three classes.

Based on the validation results, class-balanced Logistic Regression is retained
as the preferred linear classification model.

## 7. Nonlinear Classification

To test whether nonlinear interactions between order-book state, order flow,
returns, volatility, and market activity improve directional classification,
we evaluate a histogram-based gradient-boosting classifier.

Model comparison continues to use the March 19 validation set only.

In [20]:
from sklearn.ensemble import HistGradientBoostingClassifier

hgb_classifier_results = []
hgb_classifiers = {}

for asset in assets:

    train_df = splits[asset]["train"]
    val_df = splits[asset]["val"]

    train_mask = (
        train_df[feature_cols].notna().all(axis=1)
        & train_df["classification_target"].notna()
    )

    val_mask = (
        val_df[feature_cols].notna().all(axis=1)
        & val_df["classification_target"].notna()
    )

    X_train = train_df.loc[train_mask, feature_cols]
    y_train = train_df.loc[
        train_mask, "classification_target"
    ].astype(int)

    X_val = val_df.loc[val_mask, feature_cols]
    y_val = val_df.loc[
        val_mask, "classification_target"
    ].astype(int)

    model = HistGradientBoostingClassifier(
        learning_rate=0.05,
        max_iter=200,
        max_leaf_nodes=15,
        l2_regularization=1.0,
        random_state=42
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_val)

    hgb_classifiers[asset] = model

    hgb_classifier_results.append({
        "asset": asset,
        "model": "HistGradientBoosting",
        "accuracy": accuracy_score(y_val, y_pred),
        "balanced_accuracy": balanced_accuracy_score(
            y_val, y_pred
        ),
        "macro_f1": f1_score(
            y_val,
            y_pred,
            average="macro",
            zero_division=0
        )
    })

hgb_classifier_results_df = pd.DataFrame(
    hgb_classifier_results
)

display(hgb_classifier_results_df)

,asset,model,accuracy,balanced_accuracy,macro_f1
0,TSLA,HistGradientBoosting,0.447827,0.393544,0.396958
1,AMZN,HistGradientBoosting,0.385267,0.399453,0.345874
2,BRK.B,HistGradientBoosting,0.412455,0.402140,0.359527


In [21]:
classifier_comparison = pd.concat(
    [
        balanced_logistic_results_df,
        hgb_classifier_results_df
    ],
    ignore_index=True
)

display(
    classifier_comparison.sort_values(
        ["asset", "balanced_accuracy"],
        ascending=[True, False]
    )
)

,asset,model,accuracy,balanced_accuracy,macro_f1
1,AMZN,Balanced Logistic Regression,0.424913,0.427740,0.419580
4,AMZN,HistGradientBoosting,0.385267,0.399453,0.345874
2,BRK.B,Balanced Logistic Regression,0.412883,0.410076,0.407589
5,BRK.B,HistGradientBoosting,0.412455,0.402140,0.359527
0,TSLA,Balanced Logistic Regression,0.424421,0.409530,0.400614
3,TSLA,HistGradientBoosting,0.447827,0.393544,0.396958


### Nonlinear model comparison

HistGradientBoosting does not improve validation performance over the
class-balanced Logistic Regression model.

Although HistGradientBoosting achieves slightly higher raw accuracy for TSLA,
its balanced accuracy is lower. For AMZN and BRK.B, class-balanced Logistic
Regression also achieves higher balanced accuracy and substantially higher
macro F1.

Because balanced accuracy and macro F1 better reflect performance across the
three directional classes, class-balanced Logistic Regression is selected as
the final classification model for all three assets.

No further classifier selection is performed using the validation set.
The model specification and asset-specific classification thresholds are now
frozen before final evaluation on the held-out March 20 test day.

In [22]:
final_classification_results = []
final_classification_predictions = {}

for asset in assets:

    test_df = splits[asset]["test"]

    test_mask = (
        test_df[feature_cols].notna().all(axis=1)
        & test_df["classification_target"].notna()
    )

    X_test = test_df.loc[test_mask, feature_cols]
    y_test = test_df.loc[
        test_mask, "classification_target"
    ].astype(int)

    # Frozen model selected using validation performance
    model = balanced_logistic_models[asset]

    y_pred = model.predict(X_test)

    final_classification_predictions[asset] = {
        "y_true": y_test,
        "y_pred": y_pred
    }

    final_classification_results.append({
        "asset": asset,
        "model": "Balanced Logistic Regression",
        "accuracy": accuracy_score(y_test, y_pred),
        "balanced_accuracy": balanced_accuracy_score(
            y_test, y_pred
        ),
        "macro_f1": f1_score(
            y_test,
            y_pred,
            average="macro",
            zero_division=0
        )
    })

final_classification_results_df = pd.DataFrame(
    final_classification_results
)

display(final_classification_results_df)

,asset,model,accuracy,balanced_accuracy,macro_f1
0,TSLA,Balanced Logistic Regression,0.427651,0.390936,0.390849
1,AMZN,Balanced Logistic Regression,0.407561,0.409763,0.403414
2,BRK.B,Balanced Logistic Regression,0.415044,0.394855,0.392446


In [30]:
test_majority_results = []

for asset in assets:

    train_df = splits[asset]["train"]
    test_df = splits[asset]["test"]

    # Use exactly the same eligible rows as the learned classifiers
    train_mask = (
        train_df[feature_cols].notna().all(axis=1)
        & train_df["classification_target"].notna()
    )

    test_mask = (
        test_df[feature_cols].notna().all(axis=1)
        & test_df["classification_target"].notna()
    )

    train_y = train_df.loc[
        train_mask,
        "classification_target"
    ].astype(int)

    y_test = test_df.loc[
        test_mask,
        "classification_target"
    ].astype(int)

    # Majority class determined only from training data
    majority_class = train_y.value_counts().idxmax()

    y_pred = np.full(
        len(y_test),
        majority_class,
        dtype=int
    )

    test_majority_results.append({
        "asset": asset,
        "model": "ZeroR / Majority Class",
        "predicted_class": label_names[majority_class],
        "accuracy": accuracy_score(y_test, y_pred),
        "balanced_accuracy": balanced_accuracy_score(
            y_test, y_pred
        ),
        "macro_f1": f1_score(
            y_test,
            y_pred,
            average="macro",
            zero_division=0
        )
    })

test_majority_results_df = pd.DataFrame(
    test_majority_results
)

display(test_majority_results_df)

,asset,model,predicted_class,accuracy,balanced_accuracy,macro_f1
0,TSLA,ZeroR / Majority Class,UP,0.412632,0.333333,0.194734
1,AMZN,ZeroR / Majority Class,STATIONARY,0.322065,0.333333,0.162405
2,BRK.B,ZeroR / Majority Class,STATIONARY,0.419281,0.333333,0.196945


In [31]:
final_test_comparison = pd.concat(
    [
        final_classification_results_df,
        test_majority_results_df
    ],
    ignore_index=True
)

display(
    final_test_comparison.sort_values(
        ["asset", "balanced_accuracy"],
        ascending=[True, False]
    )
)

,asset,model,accuracy,balanced_accuracy,macro_f1,predicted_class
1,AMZN,Balanced Logistic Regression,0.407561,0.409763,0.403414,NaN
4,AMZN,ZeroR / Majority Class,0.322065,0.333333,0.162405,STATIONARY
2,BRK.B,Balanced Logistic Regression,0.415044,0.394855,0.392446,NaN
5,BRK.B,ZeroR / Majority Class,0.419281,0.333333,0.196945,STATIONARY
0,TSLA,Balanced Logistic Regression,0.427651,0.390936,0.390849,NaN
3,TSLA,ZeroR / Majority Class,0.412632,0.333333,0.194734,UP


## 8. Final Test-Set Evaluation

The selected class-balanced Logistic Regression models are now evaluated on
March 20, which remained completely excluded from threshold estimation,
feature selection, model selection, and hyperparameter selection.

Balanced accuracy and macro F1 are emphasized because the proportions of
DOWN, STATIONARY, and UP observations differ across assets and trading days.
A majority-class classifier therefore provides a balanced accuracy of one
third while potentially achieving a substantially higher raw accuracy.

In [25]:
for asset in assets:

    y_test = final_classification_predictions[asset]["y_true"]
    y_pred = final_classification_predictions[asset]["y_pred"]

    cm = confusion_matrix(
        y_test,
        y_pred,
        labels=[-1, 0, 1]
    )

    cm_df = pd.DataFrame(
        cm,
        index=["Actual DOWN", "Actual STATIONARY", "Actual UP"],
        columns=["Pred DOWN", "Pred STATIONARY", "Pred UP"]
    )

    print("=" * 70)
    print(asset)
    print("=" * 70)

    display(cm_df)

    print(
        classification_report(
            y_test,
            y_pred,
            labels=[-1, 0, 1],
            target_names=["DOWN", "STATIONARY", "UP"],
            digits=4,
            zero_division=0
        )
    )

TSLA


,Pred DOWN,Pred STATIONARY,Pred UP
Actual DOWN,9420,2924,7423
Actual STATIONARY,3006,1895,2785
Actual UP,7698,2915,8673


              precision    recall  f1-score   support

        DOWN     0.4681    0.4766    0.4723     19767
  STATIONARY     0.2450    0.2466    0.2458      7686
          UP     0.4594    0.4497    0.4545     19286

    accuracy                         0.4277     46739
   macro avg     0.3908    0.3909    0.3908     46739
weighted avg     0.4278    0.4277    0.4277     46739

AMZN


,Pred DOWN,Pred STATIONARY,Pred UP
Actual DOWN,5672,6446,3871
Actual STATIONARY,3554,8198,3301
Actual UP,4128,6390,5179


              precision    recall  f1-score   support

        DOWN     0.4247    0.3547    0.3866     15989
  STATIONARY     0.3897    0.5446    0.4543     15053
          UP     0.4193    0.3299    0.3693     15697

    accuracy                         0.4076     46739
   macro avg     0.4113    0.4098    0.4034     46739
weighted avg     0.4117    0.4076    0.4026     46739

BRK.B


,Pred DOWN,Pred STATIONARY,Pred UP
Actual DOWN,3953,5712,4544
Actual STATIONARY,3489,10949,5155
Actual UP,2979,5456,4493


              precision    recall  f1-score   support

        DOWN     0.3793    0.2782    0.3210     14209
  STATIONARY     0.4950    0.5588    0.5250     19593
          UP     0.3166    0.3475    0.3313     12928

    accuracy                         0.4150     46730
   macro avg     0.3970    0.3949    0.3924     46730
weighted avg     0.4105    0.4150    0.4094     46730



### Final classification observations

The final out-of-sample evaluation on March 20 indicates that the
microstructure features contain modest information about the direction of
future price movements for the selected asset-horizon pairs.

The class-balanced Logistic Regression model achieves test balanced
accuracies of approximately 0.391 for TSLA, 0.410 for AMZN, and 0.395 for
BRK.B. These values are above the 1/3 balanced-accuracy level obtained by a
majority-class classifier. Macro F1 also improves substantially over the
majority-class baseline for all three assets.

The confusion matrices show that the models do not obtain these results by
predicting only the most frequent class. All three classes are predicted,
although the difficulty of distinguishing them varies across assets. For
TSLA, DOWN and UP movements are identified more successfully than the
STATIONARY class. AMZN exhibits more balanced performance across the three
classes, while BRK.B identifies STATIONARY observations more reliably than
directional movements.

The classification results therefore provide evidence of weak but
persistent out-of-sample directional structure in the order-book features.
The predictive signal is not strong enough to imply that individual
high-frequency price movements can be predicted reliably, but performance
above the simple majority-class benchmark is observed across all three
selected asset-horizon pairs.

The final test day was used only once after the classification design and
model choice had been fixed. March 20 was not used for threshold estimation,
feature selection, hyperparameter selection, or model selection.

In [27]:
balanced_logistic_results_df

,asset,model,accuracy,balanced_accuracy,macro_f1
0,TSLA,Balanced Logistic Regression,0.424421,0.409530,0.400614
1,AMZN,Balanced Logistic Regression,0.424913,0.427740,0.419580
2,BRK.B,Balanced Logistic Regression,0.412883,0.410076,0.407589


In [28]:
from pathlib import Path

OUTPUT_DIR = Path("../outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

balanced_logistic_results_df.to_csv(
    OUTPUT_DIR / "classification_validation_results.csv",
    index=False
)

print(
    "Saved:",
    OUTPUT_DIR / "classification_validation_results.csv"
)

Saved: ../outputs/classification_validation_results.csv
